# PAQD atlas

Port of `packages/niivue/examples/vox.paqd.html`. Demonstrates PAQD volume rendering presets with a cerebellar atlas and statistical overlay.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

from ipyniivue import NiiVue

BASE_URL = "https://niivue.github.io/mono"
VOLUMES = f"{BASE_URL}/volumes"
MESHES = f"{BASE_URL}/meshes"

nv = NiiVue(
    background_color=[0.1, 0.1, 0.2, 1.0],
    is_3d_crosshair_visible=True,
    is_colorbar_visible=True,
    show_render=1,
    volume_illumination=0.4,
    backend="webgl2",
)

paqd_presets = {
    "rim": [0.2, 0.7, 0.9, 0.4],
    "opaque": [0.01, 0.5, 0.5, 1.0],
    "translucent": [0.01, 0.5, 0.25, 0.4],
}

paqd = widgets.Dropdown(
    options=[("Rim", "rim"), ("Opaque", "opaque"), ("Translucent", "translucent")],
    value="rim",
    description="PAQD",
)
clip = widgets.Checkbox(value=True, description="Clip")
illumination = widgets.FloatSlider(value=0.4, min=0.0, max=1.0, step=0.05, description="Light")


def update_paqd(change=None):
    nv.volume_paqd_uniforms = paqd_presets[paqd.value]
    nv.update_gl_volume()


def update_clip(change=None):
    nv.set_clip_plane([0.0, 180, 30] if clip.value else [2.0, 180, 30])


def update_illumination(change):
    nv.volume_illumination = change["new"]


paqd.observe(update_paqd, names="value")
clip.observe(update_clip, names="value")
illumination.observe(update_illumination, names="value")

controls = widgets.HBox([paqd, clip, illumination])
display(controls)
display(nv)

nv.load_volumes(
    [
        {"url": f"{VOLUMES}/MNI152NLin6AsymC.nii.gz"},
        {"url": f"{VOLUMES}/atl-Anatom.nii.gz"},
        {"url": f"{VOLUMES}/spmMotor.nii.gz", "colormap": "warm", "calMin": 4, "calMax": 8},
    ]
)
nv.set_colormap_label_from_url(
    1,
    "https://niivue.github.io/niivue-demo-images/Cerebellum/atl-Anatom.json",
)
nv.set_volume(0, {"isColorbarVisible": False})
nv.set_volume(1, {"isColorbarVisible": False})
update_paqd()
update_clip()